In [ ]:
import socket
import cv2
import numpy as np
import time

from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
btns = base.btns_gpio
leds = base.leds

### Functions

In [ ]:
def bgr_to_hsv(color):
    bgr = np.uint8([[color]])   # shape (1,1,3)
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    return tuple(hsv[0][0])

In [ ]:
def classify_season(hsv_colors):

    skin_h, skin_s, skin_v = hsv_colors[0]
    hair_h, hair_s, hair_v = hsv_colors[1]
    eye_h, eye_s, eye_v = hsv_colors[2]

    # undertone
    if skin_h < 25:
        undertone = "warm"
    elif skin_h < 35:
        undertone = "neutral"
    else:
        undertone = "cool"

    # contrast
    contrast = (abs(skin_v-hair_v) + abs(skin_v-eye_v)) / 2

    if contrast < 40:
        contrast_type = "low"
    elif contrast < 80:
        contrast_type = "medium"
    else:
        contrast_type = "high"

    # make a decision
    if undertone == "warm":
        if contrast_type == "low":
            season = "spring"
        else:
            season = "autumn"
    else:
        if contrast_type == "low":
            season = "summer"
        else:
            season = "winter"

    return season

In [ ]:
 def all_onboard_leds_off():
    for a in leds:
        a.off()

### Main

In [ ]:
# skin, hair, eye in BGR format
colors = []
all_onboard_leds_off()

CLIENT_IP = "192.168.0.207"
LISTENING_PORT = 8888

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

sock.bind((CLIENT_IP, LISTENING_PORT))

sock.listen(1)

conn, addr = sock.accept()
print(f"Accepted connection from {addr}")

while True:
    data = conn.recv(1024)
    bgr_tuple = tuple(map(int, s.split()))
    colors.append(bgr_tuple)
    if not data:
        print("Client disconneted")
        break
    print("Received message: ", data.decode('utf8'))

conn.close()

# skin, hair, eye in HSV format
hsv_colors = []

for color in colors:
    hsv_colors.append(bgr_to_hsv(color))
    
season = classify_season(hsv_colors)
print(season)

led_num = -1

if season == 'spring':
    led_num = 0
elif season == 'summer':
    led_num = 1
elif season == 'autumn':
    led_num = 2
elif season == 'winter':
    led_num = 3
else: pass

for i in range(4):
    leds[led_num].on()
    time.sleep(0.5)
    leds[led_num].off()
    time.sleep(0.5)

leds[led_num].on()

print("Done.")
 